In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:
base_url = "https://books.toscrape.com/catalogue/"
headers = {
    "User-Agent":"Mozilla/5.0"
    
}
all_data = []

In [5]:
page_urls =[]
for i in range(1,51):
    page_urls.append(f"{base_url}page-{i}.html")

len(page_urls)

50

In [9]:
def extract_book_data(book):
    title = book.find("h3").a["title"]
    price = book.find("p",class_="price_color").text.replace("£", "")
    rating = book.find("p",class_="star-rating")["class"][1]
    availability = book.find("p",class_="instock availability").text.strip()
    product_link = urljoin(base_url, book.find("h3").a["href"])

    try:
        detail_page = requests.get(product_link, headers=headers)
        detail_soup = BeautifulSoup(detail_page.text, "html.parser")

        breadcrumb = detail_soup.find("ul", class_="breadcrumb")
        if breadcrumb:
            category = breadcrumb.find_all("a")[2].text
        else:
            category = "Unknown"

    except:
        category = "Unknown"

    return [title, category, price, rating, availability]


In [10]:
for page_url in page_urls:
    print("Scraping:", page_url)

    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        book_info = extract_book_data(book)
        all_data.append(book_info)
        time.sleep(0.2)   

    time.sleep(0.5)  

Scraping: https://books.toscrape.com/catalogue/page-1.html
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html
Scraping: https://books.toscrape.com/catalogue/page-6.html
Scraping: https://books.toscrape.com/catalogue/page-7.html
Scraping: https://books.toscrape.com/catalogue/page-8.html
Scraping: https://books.toscrape.com/catalogue/page-9.html
Scraping: https://books.toscrape.com/catalogue/page-10.html
Scraping: https://books.toscrape.com/catalogue/page-11.html
Scraping: https://books.toscrape.com/catalogue/page-12.html
Scraping: https://books.toscrape.com/catalogue/page-13.html
Scraping: https://books.toscrape.com/catalogue/page-14.html
Scraping: https://books.toscrape.com/catalogue/page-15.html
Scraping: https://books.toscrape.com/catalogue/page-16.html
Scraping: https://books.toscrape.com/catalogue/pa

In [11]:
df = pd.DataFrame(
    all_data,
    columns=["Title", "Category", "Price", "Rating", "Availability"]
)

df.shape

(1000, 5)

In [12]:
df.to_csv("books_1000_rows.csv", index=False)
print("Total rows collected:", len(df))

Total rows collected: 1000
